In [1]:
from GNN import GATv2SequenceModel
from dataset_processing import DatasetProcessing
import json
import networkx as nx
from torch_geometric.utils import to_networkx
import random
import torch
from torch.utils.data import DataLoader
import os
import gc
import pandas as pd

In [2]:

dataset_process=DatasetProcessing()
asd=[]
diras=os.listdir("data")
for i in diras:
    with open(f"data/{i}","r") as f:
        asd.append(json.load(f))
a,label_list=dataset_process.modified(asd)
a=a[35:]
sequence_list,sequence_labels=dataset_process.create_sequence_and_labels(a,label_list,5)

WORKING DIRECTORY::::f:\OE_Szakdoga\OE_Szakdolgozat\src\models\neural_networks\working


In [3]:
classes=[]
turns=[]
lane=[]
for i in sequence_labels:
    classes.append(i[0])
    turns.append(i[1][0])
    lane.append(i[1][1])

data = {
    'classes': classes,
    'turns': turns,
    'lane': lane
}

# Create a DataFrame from the dictionary
df = pd.DataFrame(data)

for column in df.columns:
    print(f"Unique value counts in column '{column}':")
    print(df[column].value_counts())
    print()

Unique value counts in column 'classes':
classes
2    1969
3    1902
0    1241
1    1199
Name: count, dtype: int64

Unique value counts in column 'turns':
turns
0    6100
1     211
Name: count, dtype: int64

Unique value counts in column 'lane':
lane
0    5336
1     975
Name: count, dtype: int64



In [4]:
dataset=[]
for graphs,labels in zip(sequence_list,sequence_labels):
    dataset.append((graphs,[labels[0],labels[1][0],labels[1][1]]))

In [5]:
len(dataset)

6304

In [6]:
# def collate_fn(batch):
#     """
#     Custom collate function for handling sequences of graphs.
#     Input:
#         batch: List of tuples (sequence of graphs, label)
#     Output:
#         sequences: List of sequences (each sequence is a list of graphs)
#         labels: Tensor of labels
#     """
#     sequences, labels, bool_labels = zip(*batch)
#     return (
#         list(sequences),
#         torch.tensor(labels, dtype=torch.long),
#         torch.tensor(bool_labels, dtype=torch.float),
#     )


# train_loader = DataLoader(
#     dataset=dataset, batch_size=1, shuffle=True, collate_fn=collate_fn
# )

In [7]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [8]:


# ELAVULT




# in_channels = 13
# hidden_channels = 16
# out_channels = 11
# edge_dim = 1
# heads = 1
# num_epochs = 10
# learning_rate = 0.001

# model = GATv2SequenceModel(
#     in_channels=in_channels,
#     hidden_channels=hidden_channels,
#     edge_dim=edge_dim,
#     out_channels=out_channels,
#     heads=heads,
# ).to("cuda")

# criterion = torch.nn.CrossEntropyLoss()
# criterion_bools=torch.nn.BCEWithLogitsLoss()
# optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


# for epoch in range(num_epochs):
#     model.train()
#     total_loss = 0

#     for sequences, labels, bool_labels in train_loader:
#         # Move data to GPU if available
#         sequences = [[graph.to("cuda") for graph in seq] for seq in sequences]
#         print(sequences)
#         labels = labels.to("cuda")
#         bool_labels=bool_labels.to("cuda")
#         print(f"Sequences shape (before model): {len(sequences)}")  # Debugging
#         outputs,bools = model(sequences)

#         print(f"Outputs shape: {outputs.shape}")  # Debugging
#         print(f"Labels shape: {labels.shape}")    # Debugging

#         # Compute loss
#         loss_class = criterion(outputs, labels)
#         loss_bool =criterion_bools(bools,bool_labels)
#         loss=loss_class+loss_bool

#         # Backward pass and optimization
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()

#     avg_loss = total_loss / len(train_loader)
#     print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

In [9]:
# torch.save(model.state_dict(),"trained.pth")

In [10]:

# HOW TO USE MODELLLLLL
# from torch_geometric.data import Batch

# model=GATv2SequenceModel(13,16,1,11)
# batched_graph=Batch.from_data_list(sequence_list[0])
# # Step 2: Extract the graph data
# x = batched_graph.x                  # Node features
# edge_index = batched_graph.edge_index  # Edge indices
# edge_attr = batched_graph.edge_attr    # Edge attributes
# batch = batched_graph.batch            # Batch vector (indicates which graph each node belongs to)

# # Step 3: Sequence length (number of graphs in this sequence)
# sequence_length = [len(sequence_list[0])]  # Only one sequence, so its length is the full sequence

# # Step 4: Pass through the model
# out, out_bool = model(x, edge_index, edge_attr, batch, sequence_length)

In [11]:
from torch_geometric.data import Batch
import torch

def collate_batch(data_list):
    """
    Custom collate function to batch sequences of graphs.
    
    Args:
        data_list: List of tuples [(sequence_of_graphs, sequence_label), ...].
                   Each sequence_of_graphs is a list of PyG Data objects.
                   Each sequence_label is the corresponding label for the sequence.
    
    Returns:
        batched_graphs: Batched PyG graphs.
        sequence_lengths: List of sequence lengths in the batch.
        labels: Tensor of sequence labels.
    """
    all_graphs = []  # Flatten all graphs in the batch
    sequence_lengths = []  # Store the length of each sequence
    labels = []  # Store the labels for each sequence

    for sequence_of_graphs, label in data_list:
        all_graphs.extend(sequence_of_graphs)  # Add all graphs in the sequence
        sequence_lengths.append(len(sequence_of_graphs))  # Length of the sequence
        labels.append(label)
    
    # Batch all graphs using PyG's Batch class
    batched_graphs = Batch.from_data_list(all_graphs)

    # Convert everything to tensors
    sequence_lengths = torch.tensor(sequence_lengths, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)

    return batched_graphs, sequence_lengths, labels

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import random_split
import copy
import numpy as np

num_epochs=60
device="cuda"
in_channels = 13
hidden_channels = 16
out_channels = 4
edge_dim = 1
heads = 2
learning_rate = 0.001
batch_size=32

model = GATv2SequenceModel(
    in_channels=in_channels,
    hidden_channels=hidden_channels,
    edge_dim=edge_dim,
    out_channels=out_channels,
    heads=heads,
).to("cuda")

data_loader=DataLoader(dataset,batch_size=batch_size,collate_fn=collate_batch,shuffle=False)
classification_loss_fn = torch.nn.CrossEntropyLoss()  # For multi-class classification
bool_loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=5, verbose=True
)


# Split dataset into training and validation sets
train_size = int(0.8 * len(dataset))  # 80% for training
val_size = len(dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoaders for training and validation
train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collate_batch, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=collate_batch, shuffle=False)

# Early stopping parameters
patience = 2500  # Number of epochs to wait before stopping
best_val_loss = float('inf')  # Initialize with infinity
best_model_state = None  # To save the best model
no_improvement_epochs = 0  # Counter for epochs without improvement

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0
    for batched_graphs, sequence_lengths, labels in train_loader:
        batched_graphs = batched_graphs.to(device)
        sequence_lengths = sequence_lengths.to("cpu").tolist()
        labels = labels.to(device)

        class_labels = labels[:, 0]
        bool_labels = labels[:, 1:]

        out, out_bool = model(
            x=batched_graphs.x,
            edge_index=batched_graphs.edge_index,
            edge_attr=batched_graphs.edge_attr,
            batch=batched_graphs.batch,
            sequence_lengths=sequence_lengths,
        )

        class_loss = classification_loss_fn(out, class_labels)
        bool_loss = bool_loss_fn(out_bool, bool_labels.float())
        total_loss = 0.7*class_loss+0.3*bool_loss

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        train_loss += total_loss.item()

    train_loss /= len(train_loader)

    # Validation phase
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

    # Validation phase
    model.eval()  # Set model to evaluation mode
    val_loss = 0
    correct_classifications = 0
    total_samples = 0

    # Metrics for boolean predictions
    bool_true_labels = []
    bool_pred_labels = []

    with torch.no_grad():
        for batched_graphs, sequence_lengths, labels in val_loader:
            batched_graphs = batched_graphs.to(device)
            sequence_lengths = sequence_lengths.to("cpu").tolist()
            labels = labels.to(device)

            class_labels = labels[:, 0]  # Multi-class classification labels
            bool_labels = labels[:, 1:]  # Multi-label binary classification labels

            out, out_bool = model(
                x=batched_graphs.x,
                edge_index=batched_graphs.edge_index,
                edge_attr=batched_graphs.edge_attr,
                batch=batched_graphs.batch,
                sequence_lengths=sequence_lengths,
            )

            # Compute losses
            class_loss = classification_loss_fn(out, class_labels)
            bool_loss = bool_loss_fn(out_bool, bool_labels.float())
            total_loss = 0.7*class_loss+0.3*bool_loss

            val_loss += total_loss.item()

            # Accuracy calculation for classification task
            _, predicted_classes = torch.max(out, dim=1)
            correct_classifications += (predicted_classes == class_labels).sum().item()
            total_samples += class_labels.size(0)

            # Boolean task predictions (multi-label binary classification)
            bool_pred_probs = torch.sigmoid(out_bool)  # Apply sigmoid to logits
            bool_pred_binary = (bool_pred_probs > 0.5).long()  # Threshold at 0.5

            # Collect true and predicted labels for boolean task metrics
            bool_true_labels.append(bool_labels.cpu().numpy())
            bool_pred_labels.append(bool_pred_binary.cpu().numpy())

    # Average validation loss
    val_loss /= len(val_loader)
    scheduler.step(val_loss)

    # Classification accuracy for multi-class task
    val_accuracy = correct_classifications / total_samples

    # Compute boolean prediction metrics
    bool_true_labels = np.vstack(bool_true_labels)  # Convert list of arrays to single array
    bool_pred_labels = np.vstack(bool_pred_labels)  # Convert list of arrays to single array

    bool_accuracy = accuracy_score(bool_true_labels, bool_pred_labels)

    # Print metrics
    print(f"Epoch {epoch + 1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, "
        f"Class Accuracy: {val_accuracy:.4f}, Bool Accuracy: {bool_accuracy:.4f}, ")

    # Early stopping logic
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())  # Save best model state
        no_improvement_epochs = 0  # Reset counter
    else:
        print(f"no imporvement: val loss:{val_loss} best loss: {best_val_loss}")
        no_improvement_epochs += 1

    if no_improvement_epochs >= patience:
        print(f"Early stopping triggered after {epoch + 1} epochs")
        break

# Load the best model state before exiting
model.load_state_dict(best_model_state)

f:\OE_Szakdoga\OE_Szakdolgozat\myenv\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1, Train Loss: 1.0162, Val Loss: 0.9757, Class Accuracy: 0.4211, Bool Accuracy: 0.8311, 
Epoch 2, Train Loss: 0.9141, Val Loss: 0.9468, Class Accuracy: 0.4409, Bool Accuracy: 0.8311, 
Epoch 3, Train Loss: 0.8915, Val Loss: 0.9771, Class Accuracy: 0.4298, Bool Accuracy: 0.8311, 
no imporvement: val loss:0.9771436676383018 best loss: 0.9468210145831109
Epoch 4, Train Loss: 0.8741, Val Loss: 0.9602, Class Accuracy: 0.4615, Bool Accuracy: 0.8311, 
no imporvement: val loss:0.9602428942918777 best loss: 0.9468210145831109
Epoch 5, Train Loss: 0.8579, Val Loss: 0.9638, Class Accuracy: 0.4449, Bool Accuracy: 0.8311, 
no imporvement: val loss:0.9638065278530121 best loss: 0.9468210145831109
Epoch 6, Train Loss: 0.8467, Val Loss: 0.9039, Class Accuracy: 0.4790, Bool Accuracy: 0.8311, 
Epoch 7, Train Loss: 0.8348, Val Loss: 0.9459, Class Accuracy: 0.4615, Bool Accuracy: 0.8311, 
no imporvement: val loss:0.9458689823746681 best loss: 0.9038989976048469
Epoch 8, Train Loss: 0.8408, Val Loss: 

<All keys matched successfully>

In [20]:
torch.save(model.state_dict(),"trained_60.pth")

In [4]:
model = GATv2SequenceModel(13, 16, 1, 4, 2)
model.load_state_dict(torch.load("trained_60.pth"))
model.eval()

C:\Users\Armin\AppData\Local\Temp\ipykernel_24872\1936876141.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("trained_60.pth"))


GATv2SequenceModel(
  (gat1): GATv2Conv(13, 16, heads=2)
  (gat2): GATv2Conv(32, 32, heads=2)
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (lstm): LSTM(64, 16, batch_first=True)
  (fc): Linear(in_features=16, out_features=4, bias=True)
  (fc_bools): Linear(in_features=16, out_features=2, bias=True)
)

In [7]:
from api_request_types.model import Predicter
from torchviz import make_dot
a=Predicter("trained_60.pth")
y=a.predict(sequence_list[0])
make_dot(y[0].mean(),params=dict(model.named_parameters())).render("GNN-LSTM",format="png")

f:\OE_Szakdoga\OE_Szakdolgozat\src\models\neural_networks\working\model.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path))


'GNN-LSTM.png'

In [ ]:
from torchviz import make_dot
model.eval()
model()